# ANN Classification

In [1]:
# date fruit dataset

In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("DateFruit_Dataset.csv")
df.head() # 34 features and 1 output

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [6]:
X = df.drop("Class", axis=1)
y = df["Class"]

In [7]:
y.unique()

array(['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY'],
      dtype=object)

In [10]:
# scale + encode
from sklearn.preprocessing import StandardScaler, LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

In [11]:
# split dataset
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,y, test_size=0.2, random_state=42
)

In [12]:
# scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ANN

In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [20]:
# create tensors

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [21]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [23]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [26]:
# build our model
class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model = nn.Sequential(

            # 1st hidden layer
            nn.Linear(X.shape[1], 64),
            nn.ReLU(),

            # 2nd hidden layer
            nn.Linear(64,64),
            nn.ReLU(),

            #op layer
            nn.Linear(64, 7),
            # soft max applied by loss func bydefault
        )

    def forward(self,x):
       return self.model(x)

In [27]:
model = ANN()

# loss and optimizer
criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [29]:
# training the neural network

epochs = 100
for epoch in range(epochs):
    model.train()

    running_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()
        
        outputs = model(xb)
        loss = criteria(outputs, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    train_loss = running_loss/len(train_loader)
    print(f" epoch {epoch+1}/{epochs} : loss = {train_loss}")
        

 epoch 1/100 : loss = 1.7019753145134968
 epoch 2/100 : loss = 1.0621711896813435
 epoch 3/100 : loss = 0.6931282476238583
 epoch 4/100 : loss = 0.5318883398304814
 epoch 5/100 : loss = 0.448052528111831
 epoch 6/100 : loss = 0.3883528702932855
 epoch 7/100 : loss = 0.35405518049779144
 epoch 8/100 : loss = 0.3196453244789787
 epoch 9/100 : loss = 0.28186594792034314
 epoch 10/100 : loss = 0.2598822084457978
 epoch 11/100 : loss = 0.2432637350714725
 epoch 12/100 : loss = 0.23870543422906296
 epoch 13/100 : loss = 0.21392515852399493
 epoch 14/100 : loss = 0.20548643398544061
 epoch 15/100 : loss = 0.20360028581774753
 epoch 16/100 : loss = 0.19577239680549371
 epoch 17/100 : loss = 0.17248119494837263
 epoch 18/100 : loss = 0.17282881846894388
 epoch 19/100 : loss = 0.16449992909379627
 epoch 20/100 : loss = 0.15383564956162288
 epoch 21/100 : loss = 0.14906175985284473
 epoch 22/100 : loss = 0.14583577053702396
 epoch 23/100 : loss = 0.14436530969713046
 epoch 24/100 : loss = 0.13882

In [39]:
# evaluate

model.eval()

total = 0
correct = 0

with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb)
        _, predicted =  torch.max(outputs,1) # it returns val, index

        correct += (predicted == yb).sum().item()
        total += yb.size(0) # actual sample in each batch

print("Total values : ", total)
print("Correct values : ", correct)

print("accuracy : ", (correct/total)*100 ,"%")

Total values :  180
Correct values :  168
accuracy :  93.33333333333333 %


In [49]:
from sklearn.metrics import accuracy_score, classification_report

model.eval()
with torch.no_grad():
     outputs = model(X_test_tensor)
     _, y_pred = torch.max(outputs,1) #max wala index -> prediction class

print("Accuracy score : ", accuracy_score(y_test, y_pred))
print("Classification report : \n", classification_report(y_test, y_pred))

Accuracy score :  0.9333333333333333
Classification report : 
               precision    recall  f1-score   support

           0       0.92      0.92      0.92        12
           1       0.77      0.85      0.81        20
           2       1.00      0.90      0.95        50
           3       0.80      0.80      0.80        10
           4       1.00      0.97      0.99        35
           5       1.00      1.00      1.00        33
           6       0.83      1.00      0.91        20

    accuracy                           0.93       180
   macro avg       0.90      0.92      0.91       180
weighted avg       0.94      0.93      0.93       180

